In [ ]:
from huggingface_hub import notebook_login

# 실행하면 토큰 입력창이 뜹니다.
notebook_login()

# HP-XNT / JIN-KV 3B Paper Experiments — Clean Version

이 노트북은 **논문용 3B 실험과 그래프 출력**만 남긴 clean 버전입니다.

대상 모델:

```text
meta-llama/Llama-3.2-3B
```

실행 순서가 꼬이지 않도록 아래 순서로 구성했습니다.

```text
1. 설치 / import / 실험 설정
2. 모델 로드
3. KV cache 추출
4. 모델 구조 summary
5. INT4 / HP-XNT codec 정의
6. Triton online attention kernel 정의
7. all-layer × all-KV-head paper sweep
8. summary table 생성
9. 논문용 그래프 저장
10. sequence length sweep
11. 논문용 결과 문장 자동 출력
```

주의:

```text
이 노트북은 attention-level 논문 실험입니다.
HuggingFace generate 전체 patch가 아니라, KV-cache compression + compressed online attention 성능을 측정합니다.
```

In [ ]:
!pip -q install -U transformers accelerate datasets triton

import os, math, time, warnings, heapq
from dataclasses import dataclass
from typing import Dict, Any, Optional, List, Tuple

import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

import triton
import triton.language as tl

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("triton:", triton.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")

# 원하는 repo id로 바꾸면 됨.
# 네가 실제로 사용한 Llama 3.1 1B repo id가 따로 있으면 여기에 넣어.
MODEL_ID = "meta-llama/Llama-3.2-3B"

# fallback은 필요할 때만 사용.
FALLBACK_MODEL_ID = None

SEQ_LEN = 512  # 3B paper main seq length. A100이면 1024/2048도 가능.

# Fast search config.
# 더 좋은 압축률이 필요하면 WIDE_SEARCH=True.
WIDE_SEARCH = False
if WIDE_SEARCH:
    GROUP_SIZES = (16, 32, 64, 128)
    M_LIST = (4, 8, 16, 32)
else:
    GROUP_SIZES = (64, 128)
    M_LIST = (8, 16, 32)

ITERS = 4
RESTARTS = 1

print("MODEL_ID:", MODEL_ID)
print("GROUP_SIZES:", GROUP_SIZES, "M_LIST:", M_LIST, "ITERS:", ITERS, "RESTARTS:", RESTARTS)

# ============================================================
# Paper experiment settings
# ============================================================

PAPER_OUT_DIR = "hpxnt_3b_paper_outputs"
os.makedirs(PAPER_OUT_DIR, exist_ok=True)

# 논문 최종 수치는 80~200 권장. 빠른 디버그면 낮춰도 됨.
PAPER_MAIN_REPEATS = 80
PAPER_AUTOTUNE_REPEATS = 30
PAPER_BT_LIST = (16, 32, 64, 128)

# True: all layers × all KV heads
RUN_ALL_PAIRS = True
MAX_PAIRS_DEBUG = None  # 예: 8, 16으로 두면 빠른 디버그

def savefig_clean(name):
    path_png = os.path.join(PAPER_OUT_DIR, name + ".png")
    path_pdf = os.path.join(PAPER_OUT_DIR, name + ".pdf")
    plt.tight_layout()
    plt.savefig(path_png, dpi=220, bbox_inches="tight")
    plt.savefig(path_pdf, bbox_inches="tight")
    print("saved:", path_png)
    print("saved:", path_pdf)

print("PAPER_OUT_DIR:", PAPER_OUT_DIR)

In [ ]:
# ============================================================
# Load model
# ============================================================

def load_model_or_fallback(model_id, fallback_id=None, dtype=None):
    if dtype is None:
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    candidates = [model_id]
    if fallback_id and fallback_id != model_id:
        candidates.append(fallback_id)

    last_err = None
    for mid in candidates:
        try:
            print(f"Trying model: {mid}")
            tok = AutoTokenizer.from_pretrained(mid)
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token

            mdl = AutoModelForCausalLM.from_pretrained(
                mid,
                torch_dtype=dtype,
                device_map="auto" if torch.cuda.is_available() else None,
            )
            mdl.eval()
            print("loaded:", mid)
            return mid, tok, mdl
        except Exception as e:
            last_err = e
            print("FAILED:", mid)
            print(type(e).__name__, str(e)[:800])

    raise RuntimeError(f"Could not load model. Last error: {last_err}")

LOADED_MODEL_ID, tokenizer, model = load_model_or_fallback(MODEL_ID, FALLBACK_MODEL_ID)

cfg = model.config
print("LOADED_MODEL_ID:", LOADED_MODEL_ID)
print("hidden_size:", getattr(cfg, "hidden_size", None))
print("num_hidden_layers:", getattr(cfg, "num_hidden_layers", None))
print("num_attention_heads:", getattr(cfg, "num_attention_heads", None))
print("num_key_value_heads:", getattr(cfg, "num_key_value_heads", None))
print("head_dim:", getattr(cfg, "head_dim", None) or getattr(cfg, "hidden_size", 0) // getattr(cfg, "num_attention_heads", 1))

In [ ]:
# ============================================================
# Extract actual post-RoPE Llama KV cache
# Robust to DynamicCache / legacy tuple formats
# ============================================================

prompt_parts = [
    "In a quiet village near the mountains, a young engineer tried to build a small machine ",
    "that could remember every story told by the villagers. The machine was simple at first, ",
    "but as more stories were added, it needed a smarter way to store memories. "
]
prompt = "".join(prompt_parts) * 64
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=SEQ_LEN).to(model.device)

with torch.no_grad():
    out = model(**inputs, use_cache=True, output_hidden_states=False, return_dict=True)

def cache_to_layer_kv(cache):
    if isinstance(cache, (tuple, list)):
        if len(cache) > 0 and isinstance(cache[0], (tuple, list)) and len(cache[0]) >= 2:
            return [(layer[0], layer[1]) for layer in cache]

    if hasattr(cache, "to_legacy_cache"):
        try:
            legacy = cache.to_legacy_cache()
            if isinstance(legacy, (tuple, list)) and len(legacy) > 0:
                if isinstance(legacy[0], (tuple, list)) and len(legacy[0]) >= 2:
                    return [(layer[0], layer[1]) for layer in legacy]
        except Exception as e:
            print("to_legacy_cache failed:", repr(e))

    if hasattr(cache, "key_cache") and hasattr(cache, "value_cache"):
        return list(zip(cache.key_cache, cache.value_cache))

    if hasattr(cache, "layers"):
        out_layers = []
        for i, layer in enumerate(cache.layers):
            K = None
            V = None
            for k_name in ["keys", "key_cache", "key_states"]:
                if hasattr(layer, k_name):
                    K = getattr(layer, k_name)
                    break
            for v_name in ["values", "value_cache", "value_states"]:
                if hasattr(layer, v_name):
                    V = getattr(layer, v_name)
                    break
            if K is None or V is None:
                attrs = [a for a in dir(layer) if not a.startswith("_")]
                raise TypeError(f"Cannot find keys/values on cache layer {i}. attrs sample={attrs[:40]}")
            out_layers.append((K, V))
        return out_layers

    attrs = [a for a in dir(cache) if not a.startswith("_")]
    raise TypeError(f"Unsupported cache type={type(cache)}, attrs sample={attrs[:50]}")

pkv_layers = cache_to_layer_kv(out.past_key_values)

def normalize_cache_layout(K, V):
    # expected [B,H,T,D]. Some backends may be [B,T,H,D].
    if K.ndim != 4 or V.ndim != 4:
        raise ValueError(f"Expected 4D K/V cache, got K={K.shape}, V={V.shape}")
    if K.shape[1] > K.shape[2] and K.shape[2] <= 128:
        K = K.transpose(1, 2).contiguous()
        V = V.transpose(1, 2).contiguous()
    return K, V

K0, V0 = normalize_cache_layout(pkv_layers[0][0], pkv_layers[0][1])
num_layers = len(pkv_layers)
num_kv_heads = K0.shape[1]
T_cache = K0.shape[2]
D_head = K0.shape[3]

def get_cache_kv(layer_idx, kv_head_idx):
    K, V = normalize_cache_layout(pkv_layers[layer_idx][0], pkv_layers[layer_idx][1])
    return K[0, kv_head_idx].contiguous().half(), V[0, kv_head_idx].contiguous().half()

print("cache object type:", type(out.past_key_values))
print("num_layers:", num_layers)
print("num_kv_heads:", num_kv_heads)
print("T_cache:", T_cache)
print("D_head:", D_head)
print("layer0 K normalized shape:", K0.shape)

## 1. Model / Cache Summary

이 셀은 **모델 로드와 KV cache 추출 이후**에 실행됩니다.  
이전 에러가 났던 `loaded_model_id`, `pkv_legacy` 문제를 없앤 완전 방어형 summary입니다.

In [ ]:
# ============================================================
# Robust model/config/cache summary
# Must run after model load and cache extraction.
# ============================================================

assert "model" in globals(), "먼저 model load 셀을 실행해야 합니다."
assert "tokenizer" in globals(), "먼저 tokenizer load 셀을 실행해야 합니다."
assert "pkv_layers" in globals(), "먼저 KV cache extraction 셀을 실행해야 합니다."

cfg = model.config

MODEL_NAME_FOR_REPORT = globals().get("LOADED_MODEL_ID", getattr(cfg, "_name_or_path", MODEL_ID))

N_LAYERS_CFG = int(getattr(cfg, "num_hidden_layers", len(model.model.layers)))
N_Q_HEADS = int(getattr(cfg, "num_attention_heads"))
N_KV_HEADS_CFG = getattr(cfg, "num_key_value_heads", None)
if N_KV_HEADS_CFG is None:
    N_KV_HEADS_CFG = N_Q_HEADS
N_KV_HEADS_CFG = int(N_KV_HEADS_CFG)

HIDDEN_SIZE = int(getattr(cfg, "hidden_size"))
HEAD_DIM_CFG = getattr(cfg, "head_dim", None)
if HEAD_DIM_CFG is None:
    HEAD_DIM_CFG = HIDDEN_SIZE // N_Q_HEADS
HEAD_DIM_CFG = int(HEAD_DIM_CFG)

# Actual cache shape from extracted cache
K0, V0 = get_cache_kv(0, 0)
N_LAYERS_CACHE = int(num_layers)
N_KV_HEADS_CACHE = int(num_kv_heads)
HEAD_DIM_CACHE = int(K0.shape[-1])
T_CACHE_ACTUAL = int(K0.shape[0])
Q_PER_KV = max(1, N_Q_HEADS // N_KV_HEADS_CACHE)

# Set canonical globals used below
num_attention_heads = N_Q_HEADS
head_dim = HEAD_DIM_CACHE
T_cache = T_CACHE_ACTUAL

df_model_summary = pd.DataFrame([{
    "model_id": MODEL_NAME_FOR_REPORT,
    "seq_len": T_CACHE_ACTUAL,
    "num_layers_config": N_LAYERS_CFG,
    "num_layers_cache": N_LAYERS_CACHE,
    "num_attention_heads": N_Q_HEADS,
    "num_key_value_heads_config": N_KV_HEADS_CFG,
    "num_key_value_heads_cache": N_KV_HEADS_CACHE,
    "q_per_kv": Q_PER_KV,
    "head_dim_config": HEAD_DIM_CFG,
    "head_dim_cache": HEAD_DIM_CACHE,
    "hidden_size": HIDDEN_SIZE,
    "torch_dtype": str(next(model.parameters()).dtype),
    "device": str(next(model.parameters()).device),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
}])

display(df_model_summary)

df_model_summary.to_csv(os.path.join(PAPER_OUT_DIR, "model_summary_3b.csv"), index=False)
print("Saved:", os.path.join(PAPER_OUT_DIR, "model_summary_3b.csv"))

In [ ]:
# ============================================================
# INT4 utilities
# ============================================================

def ceil_log2(n: int) -> int:
    if n <= 1:
        return 0
    return int(math.ceil(math.log2(n)))

def ceil_log2_nonzero(n: int) -> int:
    if n <= 1:
        return 1
    return int(math.ceil(math.log2(n)))

def quantize_int4_per_channel(X: torch.Tensor, eps: float = 1e-8):
    # X: [T,D], scale per D channel
    Xf = X.float()
    max_abs = Xf.abs().amax(dim=0).clamp_min(eps)
    scale = max_abs / 7.0
    q = torch.round(Xf / scale).clamp(-8, 7).to(torch.int8)
    return q, scale.half()

def signed_int4_to_uint4(q: torch.Tensor) -> torch.Tensor:
    return (q.to(torch.int16) & 0xF).to(torch.uint8)

def uint4_to_signed_int4(u: torch.Tensor) -> torch.Tensor:
    x = (u.to(torch.int16) & 0xF)
    x = torch.where(x >= 8, x - 16, x)
    return x.to(torch.int8)

def dequant_from_uint4(u: torch.Tensor, scale: torch.Tensor, dtype=torch.float16):
    q = uint4_to_signed_int4(u).float()
    return (q * scale.float()).to(dtype)

def int4_quant_dequant(X: torch.Tensor):
    q, scale = quantize_int4_per_channel(X)
    u = signed_int4_to_uint4(q)
    return dequant_from_uint4(u, scale, dtype=X.dtype), u, scale

def uint4_to_planes(u: torch.Tensor):
    u = (u & 0xF).to(torch.uint8)
    return [(((u >> p) & 1).bool()) for p in range(4)]

def planes_to_uint4(b0, b1, b2, b3):
    u = torch.zeros_like(b0, dtype=torch.uint8)
    u |= b0.to(torch.uint8)
    u |= (b1.to(torch.uint8) << 1)
    u |= (b2.to(torch.uint8) << 2)
    u |= (b3.to(torch.uint8) << 3)
    return (u & 0xF).to(torch.uint8)

def high3_from_b1b2b3(b1, b2, b3):
    return (b1.to(torch.uint8) | (b2.to(torch.uint8) << 1) | (b3.to(torch.uint8) << 2)).to(torch.uint8)

def b1b2b3_from_high3(h):
    h = h.to(torch.uint8) & 0x7
    b1 = ((h >> 0) & 1).bool()
    b2 = ((h >> 1) & 1).bool()
    b3 = ((h >> 2) & 1).bool()
    return b1, b2, b3

In [ ]:
# ============================================================
# Residual/Huffman codecs
# ============================================================

def entropy_from_counts(counts):
    total = sum(int(c) for c in counts)
    if total <= 0:
        return 0.0
    H = 0.0
    for c in counts:
        c = int(c)
        if c:
            p = c / total
            H -= p * math.log2(p)
    return H

def huffman_lengths(counts):
    K = len(counts)
    heap = []
    uid = 0
    lengths = {i: 0 for i in range(K)}
    for i, c in enumerate(counts):
        w = int(c) if int(c) > 0 else 1
        heapq.heappush(heap, (w, uid, [i]))
        uid += 1
    if K == 1:
        return [1]
    while len(heap) > 1:
        w1, _, s1 = heapq.heappop(heap)
        w2, _, s2 = heapq.heappop(heap)
        for s in s1 + s2:
            lengths[s] += 1
        heapq.heappush(heap, (w1 + w2, uid, s1 + s2))
        uid += 1
    return [max(1, lengths[i]) for i in range(K)]

def huffman_bits_from_counts(counts, overhead_bits=0):
    lengths = huffman_lengths(counts)
    bits = int(sum(int(c) * int(l) for c, l in zip(counts, lengths)) + int(overhead_bits))
    return bits, lengths

def counts_bincount_uint(symbols: torch.Tensor, Ksym: int):
    flat = symbols.reshape(-1).to(torch.long)
    return torch.bincount(flat, minlength=Ksym).detach().cpu().tolist()

@dataclass
class SymbolResidualCodec:
    mode: str
    bits: int
    nnz: int
    density: float
    residual_symbol: torch.Tensor
    nbits: int
    meta: Dict[str, Any]

def symbol_residual_codec_fast(residual_symbol: torch.Tensor, nbits: int) -> SymbolResidualCodec:
    R = residual_symbol.to(torch.uint8).contiguous()
    T, W = R.shape
    N = T * W
    Ksym = 1 << int(nbits)
    counts = counts_bincount_uint(R, Ksym)
    nnz = int(N - counts[0])
    density = nnz / max(N, 1)

    if nnz == 0:
        return SymbolResidualCodec("all_zero", 1, 0, 0.0, R, nbits, {"counts": counts})

    candidates = []
    header = 2
    candidates.append((f"bitmap{nbits}", header + nbits*N, {"kind": "bitmap"}))

    if density < 0.55:
        bits_global = header + ceil_log2_nonzero(N + 1) + nnz * (ceil_log2_nonzero(N) + nbits)
        bits_row = header + T * ceil_log2_nonzero(W + 1) + nnz * (ceil_log2_nonzero(W) + nbits)
        candidates += [
            ("global_sparse", bits_global, {"kind": "sparse"}),
            ("row_sparse", bits_row, {"kind": "sparse"}),
        ]

    codebook_overhead = 8 * Ksym
    huff_bits, huff_lengths = huffman_bits_from_counts(counts, overhead_bits=codebook_overhead)
    candidates.append((f"prefix_huffman{Ksym}", huff_bits, {"kind": "prefix", "lengths": huff_lengths}))

    mode, bits, meta = min(candidates, key=lambda x: x[1])
    meta = dict(meta)
    meta.update({
        "counts": counts,
        "entropy_bits_est": entropy_from_counts(counts) * N,
        "all_candidate_bits": {m: int(b) for m, b, _ in candidates},
    })
    return SymbolResidualCodec(mode, int(bits), nnz, float(density), R, nbits, meta)

In [ ]:
# ============================================================
# Vectorized template fitting / Symbol codec
# ============================================================

@dataclass
class SymbolCodec:
    mode: str
    bits: int
    T: int
    D: int
    nbits: int
    meta: Dict[str, Any]
    raw: Optional[torch.Tensor] = None
    templates: Optional[List[torch.Tensor]] = None
    assigns: Optional[List[torch.Tensor]] = None
    residuals: Optional[List[SymbolResidualCodec]] = None

def assignment_codec_bits(assign: torch.Tensor, M: int):
    T = int(assign.numel())
    if M <= 1:
        return 0, "none_M1", {"counts": [T], "fixed_bits": 0, "huffman_bits": 0}

    counts = counts_bincount_uint(assign.to(torch.long), M)
    fixed_bits = T * ceil_log2(M)
    huff_bits, lengths = huffman_bits_from_counts(counts, overhead_bits=8*M)
    if huff_bits < fixed_bits:
        return int(huff_bits), "assign_huffman", {
            "counts": counts, "fixed_bits": int(fixed_bits), "huffman_bits": int(huff_bits), "lengths": lengths
        }
    return int(fixed_bits), "assign_fixed", {
        "counts": counts, "fixed_bits": int(fixed_bits), "huffman_bits": int(huff_bits), "lengths": lengths
    }

def residual_bits_for_batched(residual_b: torch.Tensor, nbits: int):
    bits = 0
    nnz = 0
    for g in range(residual_b.shape[0]):
        rc = symbol_residual_codec_fast(residual_b[g], nbits)
        bits += rc.bits
        nnz += rc.nnz
    return int(bits), int(nnz)

def fit_symbol_templates_batched(Hg: torch.Tensor, nbits: int, M: int, iters=ITERS, restarts=RESTARTS, seed=0):
    Hg = Hg.to(torch.uint8).contiguous()
    G, T, W = Hg.shape
    device = Hg.device
    Ksym = 1 << int(nbits)
    M = min(max(int(M), 1), T)
    best = None

    for r in range(restarts):
        gen = torch.Generator(device=device)
        gen.manual_seed(seed + 1009*r)

        idx = torch.randint(0, T, (G, M), generator=gen, device=device)
        templates = torch.gather(Hg, 1, idx[:, :, None].expand(G, M, W)).contiguous()

        for _ in range(iters):
            dist = (Hg[:, :, None, :] != templates[:, None, :, :]).sum(dim=-1)
            assign = dist.argmin(dim=-1)
            oh = F.one_hot(assign, num_classes=M).float()
            counts_per_template = oh.sum(dim=1)

            counts = []
            for k in range(Ksym):
                cnt = torch.einsum("gtm,gtw->gmw", oh, (Hg == k).float())
                counts.append(cnt)

            new_templates = torch.stack(counts, dim=0).argmax(dim=0).to(torch.uint8)

            empty = counts_per_template == 0
            if bool(empty.any()):
                gidx, midx = torch.where(empty)
                ridx = torch.randint(0, T, (gidx.numel(),), generator=gen, device=device)
                new_templates[gidx, midx] = Hg[gidx, ridx]

            templates = new_templates.contiguous()

        dist = (Hg[:, :, None, :] != templates[:, None, :, :]).sum(dim=-1)
        assign = dist.argmin(dim=-1)
        chosen = torch.gather(templates, 1, assign[:, :, None].expand(G, T, W))
        residual = (Hg ^ chosen).to(torch.uint8)

        residual_bits, _ = residual_bits_for_batched(residual, nbits)
        assign_bits = sum(assignment_codec_bits(assign[g], M)[0] for g in range(G))
        loss = int(residual_bits + assign_bits)

        if best is None or loss < best["loss"]:
            best = {"templates": templates, "assign": assign, "residual": residual, "loss": loss}

    return best

def compress_symbol_raw(H: torch.Tensor, nbits: int):
    T, D = H.shape
    return SymbolCodec(f"raw_symbol{nbits}", nbits*T*D, T, D, nbits, {"raw_bits": nbits*T*D}, raw=H.to(torch.uint8).contiguous())

def compress_symbol_row_template_fast(H: torch.Tensor, nbits: int, M: int, iters=ITERS, restarts=RESTARTS, seed=0):
    H = H.to(torch.uint8).contiguous()
    T, D = H.shape
    fit = fit_symbol_templates_batched(H[None, :, :], nbits, M, iters, restarts, seed)
    tmpl = fit["templates"][0]
    assign = fit["assign"][0]
    residual = fit["residual"][0]
    rc = symbol_residual_codec_fast(residual, nbits)

    M_eff = int(tmpl.shape[0])
    template_bits = M_eff * D * nbits
    assign_bits, assign_mode, _ = assignment_codec_bits(assign, M_eff)
    bits = template_bits + assign_bits + rc.bits

    return SymbolCodec(
        f"row_templates_symbol{nbits}", int(bits), T, D, nbits,
        {
            "M": M_eff, "template_bits": int(template_bits), "assignment_bits": int(assign_bits),
            "assignment_mode": assign_mode, "residual_bits": int(rc.bits), "residual_mode": rc.mode,
            "residual_nnz": int(rc.nnz), "residual_density": float(rc.density),
            "saving_vs_raw_symbol": float((nbits*T*D - bits)/max(nbits*T*D,1))
        },
        templates=[tmpl], assigns=[assign], residuals=[rc]
    )

def compress_symbol_group_template_fast(H: torch.Tensor, nbits: int, group_size: int, M: int, iters=ITERS, restarts=RESTARTS, seed=0):
    H = H.to(torch.uint8).contiguous()
    T, D = H.shape
    if D % group_size != 0:
        raise ValueError("group_size must divide D")

    G = D // group_size
    Hg = H.view(T, G, group_size).permute(1, 0, 2).contiguous()
    fit = fit_symbol_templates_batched(Hg, nbits, M, iters, restarts, seed)

    template_bits = assign_bits = residual_bits = residual_nnz = 0
    templates, assigns, residuals, assign_modes = [], [], [], []

    for g in range(G):
        tmpl = fit["templates"][g].contiguous()
        assign = fit["assign"][g].contiguous()
        residual = fit["residual"][g].contiguous()
        rc = symbol_residual_codec_fast(residual, nbits)
        M_eff = int(tmpl.shape[0])
        abits, amode, _ = assignment_codec_bits(assign, M_eff)

        template_bits += M_eff * group_size * nbits
        assign_bits += abits
        residual_bits += rc.bits
        residual_nnz += rc.nnz

        templates.append(tmpl)
        assigns.append(assign)
        residuals.append(rc)
        assign_modes.append(amode)

    bits = template_bits + assign_bits + residual_bits
    return SymbolCodec(
        f"group_templates_symbol{nbits}_g{group_size}", int(bits), T, D, nbits,
        {
            "group_size": int(group_size), "groups": int(G), "M": int(M),
            "template_bits": int(template_bits), "assignment_bits": int(assign_bits),
            "assignment_modes": assign_modes, "residual_bits": int(residual_bits),
            "residual_nnz": int(residual_nnz), "residual_density": float(residual_nnz/max(T*D,1)),
            "saving_vs_raw_symbol": float((nbits*T*D - bits)/max(nbits*T*D,1))
        },
        templates=templates, assigns=assigns, residuals=residuals
    )

def compress_symbol_auto_fast(H: torch.Tensor, nbits: int, group_sizes=GROUP_SIZES, M_list=M_LIST, iters=ITERS, restarts=RESTARTS, seed=0):
    H = H.to(torch.uint8).contiguous()
    T, D = H.shape
    cands = [compress_symbol_raw(H, nbits)]

    for M in M_list:
        cands.append(compress_symbol_row_template_fast(H, nbits, M, iters, restarts, seed+M))

    for idx_gs, gs in enumerate(group_sizes):
        if D % gs == 0:
            for idx_m, M in enumerate(M_list):
                cands.append(compress_symbol_group_template_fast(H, nbits, gs, M, iters, restarts, seed+10000+idx_gs*100+idx_m))

    best = min(cands, key=lambda c: c.bits)
    best.meta["num_candidates"] = len(cands)
    best.meta["raw_symbol_bits"] = int(nbits*T*D)
    best.meta["candidate_bits"] = {c.mode: int(c.bits) for c in cands}
    return best

def decode_symbol_fast(codec: SymbolCodec):
    if codec.mode.startswith("raw_symbol"):
        return codec.raw.to(torch.uint8)

    if codec.mode.startswith("row_templates_symbol"):
        return (codec.templates[0][codec.assigns[0]] ^ codec.residuals[0].residual_symbol).to(torch.uint8)

    if codec.mode.startswith("group_templates_symbol"):
        out_g = []
        for g in range(len(codec.templates)):
            out_g.append((codec.templates[g][codec.assigns[g]] ^ codec.residuals[g].residual_symbol).to(torch.uint8))
        return torch.cat(out_g, dim=1)

    raise ValueError(codec.mode)

In [ ]:
# ============================================================
# Fast HP-XNT-v3 block/pair
# ============================================================

@dataclass
class FastBlock:
    candidate: str
    T: int
    D: int
    u_uint4: torch.Tensor
    scale: torch.Tensor
    total_bits: int
    raw_bits: int
    effective_bits_per_scalar: float
    meta: Dict[str, Any]
    b0_raw: Optional[torch.Tensor] = None
    b0_codec: Optional[SymbolCodec] = None
    high3_codec: Optional[SymbolCodec] = None
    raw_planes: Optional[Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]] = None

def build_raw_block(u, scale):
    b0,b1,b2,b3 = uint4_to_planes(u)
    T,D = u.shape
    raw_bits = 4*T*D + 16*D
    return FastBlock(
        "raw_int4", T, D, u.detach(), scale.detach(), int(raw_bits), int(raw_bits),
        raw_bits/(T*D), {"scale_bits": 16*D},
        raw_planes=(b0.detach(), b1.detach(), b2.detach(), b3.detach())
    )

def build_high3_block(u, scale, h3_codec):
    b0,b1,b2,b3 = uint4_to_planes(u)
    T,D = u.shape
    bits = T*D + h3_codec.bits + 16*D
    raw_bits = 4*T*D + 16*D
    return FastBlock(
        "high3_joint", T, D, u.detach(), scale.detach(), int(bits), int(raw_bits), bits/(T*D),
        {"scale_bits":16*D, "b0_raw_bits":T*D, "high3_bits":int(h3_codec.bits), "high3_mode":h3_codec.mode},
        b0_raw=b0.detach(), high3_codec=h3_codec
    )

def build_b0_plus_high3_block(u, scale, b0_codec, h3_codec):
    T,D = u.shape
    bits = b0_codec.bits + h3_codec.bits + 16*D
    raw_bits = 4*T*D + 16*D
    return FastBlock(
        "b0_plus_high3", T, D, u.detach(), scale.detach(), int(bits), int(raw_bits), bits/(T*D),
        {"scale_bits":16*D, "b0_bits":int(b0_codec.bits), "b0_mode":b0_codec.mode,
         "high3_bits":int(h3_codec.bits), "high3_mode":h3_codec.mode},
        b0_codec=b0_codec, high3_codec=h3_codec
    )

def compress_block_fast_hpxnt_v3(X: torch.Tensor, seed=0):
    q, scale = quantize_int4_per_channel(X.contiguous())
    u = signed_int4_to_uint4(q)
    b0,b1,b2,b3 = uint4_to_planes(u)
    h3 = high3_from_b1b2b3(b1,b2,b3)

    raw = build_raw_block(u, scale)
    b0_codec = compress_symbol_auto_fast(b0.to(torch.uint8), nbits=1, seed=seed+101)
    h3_codec = compress_symbol_auto_fast(h3, nbits=3, seed=seed+303)

    cands = [
        raw,
        build_high3_block(u, scale, h3_codec),
        build_b0_plus_high3_block(u, scale, b0_codec, h3_codec),
    ]
    best = min(cands, key=lambda c: c.total_bits)
    best.meta["candidate_table"] = {c.candidate: c.effective_bits_per_scalar for c in cands}
    return best

def decode_block_to_uint4(block: FastBlock):
    if block.candidate == "raw_int4":
        b0,b1,b2,b3 = block.raw_planes
    elif block.candidate == "high3_joint":
        b0 = block.b0_raw
        h3 = decode_symbol_fast(block.high3_codec)
        b1,b2,b3 = b1b2b3_from_high3(h3)
    elif block.candidate == "b0_plus_high3":
        b0 = decode_symbol_fast(block.b0_codec).bool()
        h3 = decode_symbol_fast(block.high3_codec)
        b1,b2,b3 = b1b2b3_from_high3(h3)
    else:
        raise ValueError(block.candidate)

    u_hat = planes_to_uint4(b0,b1,b2,b3)
    if not torch.equal(u_hat, block.u_uint4):
        diff = (u_hat.to(torch.int16)-block.u_uint4.to(torch.int16)).abs().max().item()
        raise RuntimeError(f"Exact uint4 decode failed for {block.candidate}, max diff={diff}")
    return u_hat

def decode_block_fast(block: FastBlock, dtype=torch.float16):
    u_hat = decode_block_to_uint4(block)
    return dequant_from_uint4(u_hat, block.scale, dtype=dtype)

@dataclass
class FastPair:
    K_block: FastBlock
    V_block: FastBlock
    report: Dict[str, Any]

def compress_pair_fast_hpxnt_v3(K, V, seed=0):
    Kb = compress_block_fast_hpxnt_v3(K, seed=seed+17)
    Vb = compress_block_fast_hpxnt_v3(V, seed=seed+29)
    T,D = K.shape
    total_bits = Kb.total_bits + Vb.total_bits
    raw_bits = Kb.raw_bits + Vb.raw_bits
    bps = total_bits/(2*T*D)
    saving = raw_bits - total_bits

    return FastPair(Kb, Vb, {
        "mode":"fast_hp_xnt_v3_exact_pair",
        "T":T, "D":D,
        "pair_total_bits":int(total_bits),
        "pair_effective_bits_per_scalar":float(bps),
        "raw_INT4_inc_scale_bits":int(raw_bits),
        "raw_INT4_inc_scale_bps":float(raw_bits/(2*T*D)),
        "saving_bits_vs_INT4_inc_scale":int(saving),
        "saving_ratio_vs_INT4_inc_scale":float(saving/max(raw_bits,1)),
        "compression_vs_fp16":float(16.0/bps),
        "K_candidate":Kb.candidate, "V_candidate":Vb.candidate,
        "K_bits":float(Kb.effective_bits_per_scalar), "V_bits":float(Vb.effective_bits_per_scalar),
        "K_candidate_table":Kb.meta.get("candidate_table"), "V_candidate_table":Vb.meta.get("candidate_table"),
        "K_high3_mode":Kb.meta.get("high3_mode"), "V_high3_mode":Vb.meta.get("high3_mode"),
    })

In [ ]:
# ============================================================
# v6 KernelHandle preparation
# ============================================================

@dataclass
class HPXNTKernelHandle:
    kind: str                 # "rowlike" or "group"
    b0: torch.Tensor          # rowlike [T,D], group [T,D]
    templates: torch.Tensor   # rowlike [M,D], group [G,M,GS]
    assigns: torch.Tensor     # rowlike [T] int32, group [G,T] int32
    residual: torch.Tensor    # rowlike [T,D], group [G,T,GS]
    scale: torch.Tensor       # [D]
    T: int
    D: int
    G: int
    GS: int
    M: int
    mode: str
    candidate: str

def _is_template_block(block: FastBlock):
    return (
        block.candidate in ["high3_joint", "b0_plus_high3"]
        and block.high3_codec is not None
        and (
            block.high3_codec.mode.startswith("row_templates_symbol")
            or block.high3_codec.mode.startswith("group_templates_symbol")
        )
    )

def _decode_b0_for_handle(block: FastBlock):
    dev = block.u_uint4.device
    if block.candidate == "high3_joint":
        return block.b0_raw.to(device=dev, dtype=torch.uint8).contiguous()
    if block.candidate == "b0_plus_high3":
        return decode_symbol_fast(block.b0_codec).to(device=dev, dtype=torch.uint8).contiguous()
    raise ValueError(f"Unsupported candidate for b0 decode: {block.candidate}")

def prepare_hpxnt_handle(block: FastBlock, force_rowlike_g64=True):
    # Prepare tensors once; removes repeated stack/to/contiguous/b0-decode overhead.
    assert _is_template_block(block), (
        f"template HP-XNT block required, got candidate={block.candidate}, "
        f"mode={getattr(block.high3_codec, 'mode', None)}"
    )

    dev = block.u_uint4.device
    T, D = block.T, block.D
    codec = block.high3_codec
    b0 = _decode_b0_for_handle(block)
    scale = block.scale.to(device=dev).contiguous()

    if codec.mode.startswith("row_templates_symbol"):
        templates = codec.templates[0].to(device=dev, dtype=torch.uint8).contiguous()
        assigns = codec.assigns[0].to(device=dev, dtype=torch.int32).contiguous()
        residual = codec.residuals[0].residual_symbol.to(device=dev, dtype=torch.uint8).contiguous()
        return HPXNTKernelHandle(
            kind="rowlike", b0=b0, templates=templates, assigns=assigns, residual=residual, scale=scale,
            T=T, D=D, G=1, GS=D, M=int(templates.shape[0]), mode=codec.mode, candidate=block.candidate
        )

    if codec.mode.startswith("group_templates_symbol"):
        GS = int(codec.meta["group_size"])
        G = int(codec.meta["groups"])
        M = int(codec.templates[0].shape[0])

        # Important optimization:
        # D=64 and group_size=64 means G=1, so this is row-like.
        if force_rowlike_g64 and G == 1 and GS == D:
            templates = codec.templates[0].to(device=dev, dtype=torch.uint8).contiguous()
            assigns = codec.assigns[0].to(device=dev, dtype=torch.int32).contiguous()
            residual = codec.residuals[0].residual_symbol.to(device=dev, dtype=torch.uint8).contiguous()
            return HPXNTKernelHandle(
                kind="rowlike", b0=b0, templates=templates, assigns=assigns, residual=residual, scale=scale,
                T=T, D=D, G=1, GS=D, M=M, mode=codec.mode + "_as_rowlike", candidate=block.candidate
            )

        templates = torch.stack(
            [x.to(device=dev, dtype=torch.uint8).contiguous() for x in codec.templates],
            dim=0
        ).contiguous()
        assigns = torch.stack(
            [x.to(device=dev, dtype=torch.int32).contiguous() for x in codec.assigns],
            dim=0
        ).contiguous()
        residual = torch.stack(
            [x.residual_symbol.to(device=dev, dtype=torch.uint8).contiguous() for x in codec.residuals],
            dim=0
        ).contiguous()
        return HPXNTKernelHandle(
            kind="group", b0=b0, templates=templates, assigns=assigns, residual=residual, scale=scale,
            T=T, D=D, G=G, GS=GS, M=M, mode=codec.mode, candidate=block.candidate
        )

    raise ValueError(codec.mode)

def prepare_pair_handles(pair: FastPair):
    return prepare_hpxnt_handle(pair.K_block), prepare_hpxnt_handle(pair.V_block)

def dense_attention(Q, K, V):
    scores = (Q.float() @ K.float().transpose(0, 1)) / math.sqrt(K.shape[-1])
    probs = torch.softmax(scores, dim=-1)
    return (probs @ V.float()).to(Q.dtype)

def time_fn(fn, repeats=100, warmup=10):
    for _ in range(warmup):
        _ = fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    last = None
    for _ in range(repeats):
        last = fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / repeats, last

In [ ]:
# ============================================================
# v6 Triton online attention kernels
# ============================================================

@triton.jit
def hpxnt_rowlike_online_kernel_v6(
    Q,
    K_B0, K_TEMPL, K_ASSIGN, K_RESID, K_SCALE,
    V_B0, V_TEMPL, V_ASSIGN, V_RESID, V_SCALE,
    OUT,
    NQ: tl.constexpr, T: tl.constexpr, D: tl.constexpr,
    MK: tl.constexpr, MV: tl.constexpr,
    INV_SQRT_D: tl.constexpr,
    BLOCK_T: tl.constexpr, BLOCK_D: tl.constexpr
):
    qid = tl.program_id(0)
    offs_d = tl.arange(0, BLOCK_D)
    mask_d = offs_d < D

    q = tl.load(Q + qid * D + offs_d, mask=mask_d, other=0.0).to(tl.float32)
    k_scale = tl.load(K_SCALE + offs_d, mask=mask_d, other=0.0).to(tl.float32)
    qk_scaled = q * k_scale * INV_SQRT_D

    m_i = tl.full((), -3.4028234663852886e38, tl.float32)
    l_i = tl.full((), 0.0, tl.float32)
    acc = tl.zeros((BLOCK_D,), dtype=tl.float32)

    for start in range(0, T, BLOCK_T):
        offs_t = start + tl.arange(0, BLOCK_T)
        mask_t = offs_t < T

        t_mat = offs_t[:, None]
        d_mat = offs_d[None, :]

        k_b0 = tl.load(K_B0 + t_mat * D + d_mat, mask=mask_t[:, None] & mask_d[None, :], other=0).to(tl.int32) & 1
        k_a = tl.load(K_ASSIGN + offs_t, mask=mask_t, other=0).to(tl.int32)

        k_base = tl.load(
            K_TEMPL + k_a[:, None] * D + d_mat,
            mask=mask_t[:, None] & mask_d[None, :],
            other=0
        ).to(tl.int32) & 7
        k_res = tl.load(K_RESID + t_mat * D + d_mat, mask=mask_t[:, None] & mask_d[None, :], other=0).to(tl.int32) & 7

        k_h3 = k_base ^ k_res
        k_u = k_b0 | (k_h3 << 1)
        k_signed = tl.where(k_u >= 8, k_u - 16, k_u).to(tl.float32)

        scores = tl.sum(k_signed * qk_scaled[None, :], axis=1)
        scores = tl.where(mask_t, scores, -3.4028234663852886e38)

        m_block = tl.max(scores, axis=0)
        m_new = tl.maximum(m_i, m_block)
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(scores - m_new)
        p = tl.where(mask_t, p, 0.0)

        l_new = l_i * alpha + tl.sum(p, axis=0)
        acc = acc * alpha

        v_b0 = tl.load(V_B0 + t_mat * D + d_mat, mask=mask_t[:, None] & mask_d[None, :], other=0).to(tl.int32) & 1
        v_a = tl.load(V_ASSIGN + offs_t, mask=mask_t, other=0).to(tl.int32)

        v_base = tl.load(
            V_TEMPL + v_a[:, None] * D + d_mat,
            mask=mask_t[:, None] & mask_d[None, :],
            other=0
        ).to(tl.int32) & 7
        v_res = tl.load(V_RESID + t_mat * D + d_mat, mask=mask_t[:, None] & mask_d[None, :], other=0).to(tl.int32) & 7

        v_h3 = v_base ^ v_res
        v_u = v_b0 | (v_h3 << 1)
        v_signed = tl.where(v_u >= 8, v_u - 16, v_u).to(tl.float32)

        acc += tl.sum(v_signed * p[:, None], axis=0)
        m_i = m_new
        l_i = l_new

    v_scale = tl.load(V_SCALE + offs_d, mask=mask_d, other=0.0).to(tl.float32)
    out = (acc / l_i) * v_scale
    tl.store(OUT + qid * D + offs_d, out, mask=mask_d)


@triton.jit
def hpxnt_group_online_kernel_v6(
    Q,
    K_B0, K_TEMPL, K_ASSIGN, K_RESID, K_SCALE,
    V_B0, V_TEMPL, V_ASSIGN, V_RESID, V_SCALE,
    OUT,
    NQ: tl.constexpr, T: tl.constexpr, D: tl.constexpr,
    G_K: tl.constexpr, GS_K: tl.constexpr, M_K: tl.constexpr,
    G_V: tl.constexpr, GS_V: tl.constexpr, M_V: tl.constexpr,
    INV_SQRT_D: tl.constexpr,
    BLOCK_T: tl.constexpr, BLOCK_D: tl.constexpr
):
    qid = tl.program_id(0)
    offs_d = tl.arange(0, BLOCK_D)
    mask_d = offs_d < D

    q = tl.load(Q + qid * D + offs_d, mask=mask_d, other=0.0).to(tl.float32)

    gk_d = offs_d // GS_K
    lk_d = offs_d - gk_d * GS_K
    gv_d = offs_d // GS_V
    lv_d = offs_d - gv_d * GS_V

    k_scale = tl.load(K_SCALE + offs_d, mask=mask_d, other=0.0).to(tl.float32)
    qk_scaled = q * k_scale * INV_SQRT_D

    m_i = tl.full((), -3.4028234663852886e38, tl.float32)
    l_i = tl.full((), 0.0, tl.float32)
    acc = tl.zeros((BLOCK_D,), dtype=tl.float32)

    for start in range(0, T, BLOCK_T):
        offs_t = start + tl.arange(0, BLOCK_T)
        mask_t = offs_t < T

        t_mat = offs_t[:, None]
        d_mat = offs_d[None, :]

        k_b0 = tl.load(K_B0 + t_mat * D + d_mat, mask=mask_t[:, None] & mask_d[None, :], other=0).to(tl.int32) & 1

        k_a = tl.load(
            K_ASSIGN + gk_d[None, :] * T + t_mat,
            mask=mask_t[:, None] & mask_d[None, :],
            other=0
        ).to(tl.int32)

        k_base = tl.load(
            K_TEMPL + gk_d[None, :] * M_K * GS_K + k_a * GS_K + lk_d[None, :],
            mask=mask_t[:, None] & mask_d[None, :],
            other=0
        ).to(tl.int32) & 7

        k_res = tl.load(
            K_RESID + gk_d[None, :] * T * GS_K + t_mat * GS_K + lk_d[None, :],
            mask=mask_t[:, None] & mask_d[None, :],
            other=0
        ).to(tl.int32) & 7

        k_h3 = k_base ^ k_res
        k_u = k_b0 | (k_h3 << 1)
        k_signed = tl.where(k_u >= 8, k_u - 16, k_u).to(tl.float32)

        scores = tl.sum(k_signed * qk_scaled[None, :], axis=1)
        scores = tl.where(mask_t, scores, -3.4028234663852886e38)

        m_block = tl.max(scores, axis=0)
        m_new = tl.maximum(m_i, m_block)
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(scores - m_new)
        p = tl.where(mask_t, p, 0.0)

        l_new = l_i * alpha + tl.sum(p, axis=0)
        acc = acc * alpha

        v_b0 = tl.load(V_B0 + t_mat * D + d_mat, mask=mask_t[:, None] & mask_d[None, :], other=0).to(tl.int32) & 1

        v_a = tl.load(
            V_ASSIGN + gv_d[None, :] * T + t_mat,
            mask=mask_t[:, None] & mask_d[None, :],
            other=0
        ).to(tl.int32)

        v_base = tl.load(
            V_TEMPL + gv_d[None, :] * M_V * GS_V + v_a * GS_V + lv_d[None, :],
            mask=mask_t[:, None] & mask_d[None, :],
            other=0
        ).to(tl.int32) & 7

        v_res = tl.load(
            V_RESID + gv_d[None, :] * T * GS_V + t_mat * GS_V + lv_d[None, :],
            mask=mask_t[:, None] & mask_d[None, :],
            other=0
        ).to(tl.int32) & 7

        v_h3 = v_base ^ v_res
        v_u = v_b0 | (v_h3 << 1)
        v_signed = tl.where(v_u >= 8, v_u - 16, v_u).to(tl.float32)

        acc += tl.sum(v_signed * p[:, None], axis=0)
        m_i = m_new
        l_i = l_new

    v_scale = tl.load(V_SCALE + offs_d, mask=mask_d, other=0.0).to(tl.float32)
    out = (acc / l_i) * v_scale
    tl.store(OUT + qid * D + offs_d, out, mask=mask_d)


def hpxnt_hybrid_online_attention_v6(Q, K_handle: HPXNTKernelHandle, V_handle: HPXNTKernelHandle, BLOCK_T=32):
    assert Q.is_cuda, "Triton kernel requires CUDA"
    assert K_handle.T == V_handle.T and K_handle.D == V_handle.D

    T, D = K_handle.T, K_handle.D
    NQ = Q.shape[0]
    out = torch.empty((NQ, D), device=Q.device, dtype=torch.float16)
    BLOCK_D = triton.next_power_of_2(D)
    grid = (NQ,)

    if K_handle.kind == "rowlike" and V_handle.kind == "rowlike":
        hpxnt_rowlike_online_kernel_v6[grid](
            Q.contiguous(),
            K_handle.b0, K_handle.templates, K_handle.assigns, K_handle.residual, K_handle.scale,
            V_handle.b0, V_handle.templates, V_handle.assigns, V_handle.residual, V_handle.scale,
            out,
            NQ, T, D,
            K_handle.M, V_handle.M,
            INV_SQRT_D=1.0 / math.sqrt(D),
            BLOCK_T=BLOCK_T,
            BLOCK_D=BLOCK_D,
        )
    else:
        hpxnt_group_online_kernel_v6[grid](
            Q.contiguous(),
            K_handle.b0, K_handle.templates, K_handle.assigns, K_handle.residual, K_handle.scale,
            V_handle.b0, V_handle.templates, V_handle.assigns, V_handle.residual, V_handle.scale,
            out,
            NQ, T, D,
            K_handle.G, K_handle.GS, K_handle.M,
            V_handle.G, V_handle.GS, V_handle.M,
            INV_SQRT_D=1.0 / math.sqrt(D),
            BLOCK_T=BLOCK_T,
            BLOCK_D=BLOCK_D,
        )

    return out

def default_bt_for_handles(K_handle, V_handle):
    if K_handle.kind == "rowlike" and V_handle.kind == "rowlike":
        return 32
    return 128

print("v6 KernelHandle + hybrid online attention ready.")

## 2. Paper Benchmark Functions

각 layer/head pair에 대해 다음을 측정합니다.

```text
- effective bits/scalar
- compression ratio vs FP16
- saving vs INT4+scale
- latency: FP16 dense attention / dense INT4 predecoded / HP-XNT online
- HP-XNT output difference vs dense INT4
- HP-XNT output cosine vs dense INT4
```

In [ ]:
# ============================================================
# Paper benchmark functions
# ============================================================

def build_pair_handles(layer_idx=0, kv_head_idx=0, seed=0):
    K, V = get_cache_kv(layer_idx, kv_head_idx)
    pair = compress_pair_fast_hpxnt_v3(K, V, seed=seed + layer_idx*1000 + kv_head_idx)
    t0 = time.time()
    K_handle, V_handle = prepare_pair_handles(pair)
    prepare_sec = time.time() - t0
    return K, V, pair, K_handle, V_handle, prepare_sec

def output_metrics(out, ref):
    x = out.float().reshape(-1)
    y = ref.float().reshape(-1)
    return {
        "max_abs": float((x - y).abs().max().item()),
        "mean_abs": float((x - y).abs().mean().item()),
        "rel_l2": float(torch.linalg.norm(x - y).item() / (torch.linalg.norm(y).item() + 1e-12)),
        "cos": float(torch.nn.functional.cosine_similarity(x, y, dim=0).item()),
    }

def benchmark_pair_for_paper(
    layer_idx=0,
    kv_head_idx=0,
    n_queries=None,
    repeats=PAPER_MAIN_REPEATS,
    autotune_repeats=PAPER_AUTOTUNE_REPEATS,
    bt_list=PAPER_BT_LIST,
    seed=2026,
):
    if n_queries is None:
        n_queries = Q_PER_KV

    K, V, pair, K_handle, V_handle, prepare_sec = build_pair_handles(layer_idx, kv_head_idx, seed=seed)

    gen = torch.Generator(device=K.device)
    gen.manual_seed(seed + layer_idx * 1000 + kv_head_idx + n_queries)
    Q = torch.randn((n_queries, K.shape[-1]), device=K.device, dtype=K.dtype, generator=gen)

    K_i4 = decode_block_fast(pair.K_block, dtype=K.dtype)
    V_i4 = decode_block_fast(pair.V_block, dtype=V.dtype)

    ref_fp16 = dense_attention(Q, K, V)
    ref_int4 = dense_attention(Q, K_i4, V_i4)

    # Autotune only HP-XNT BLOCK_T
    best = None
    autotune_rows = []
    for bt in bt_list:
        sec, out = time_fn(
            lambda bt=bt: hpxnt_hybrid_online_attention_v6(Q, K_handle, V_handle, BLOCK_T=bt),
            repeats=max(10, autotune_repeats),
            warmup=5,
        )
        m_int4 = output_metrics(out, ref_int4)
        autotune_rows.append({
            "layer": layer_idx,
            "kv_head": kv_head_idx,
            "BLOCK_T": bt,
            "hpxnt_ms": sec * 1000,
            "max_diff_vs_dense_int4": m_int4["max_abs"],
            "cos_vs_dense_int4": m_int4["cos"],
        })
        if best is None or sec < best[0]:
            best = (sec, bt)

    selected_bt = int(best[1])

    methods = {
        "original_fp16_attention": lambda: dense_attention(Q, K, V),
        "dense_int4_predecoded_attention": lambda: dense_attention(Q, K_i4, V_i4),
        f"hpxnt_v6_online_BT{selected_bt}": lambda: hpxnt_hybrid_online_attention_v6(
            Q, K_handle, V_handle, BLOCK_T=selected_bt
        ),
    }

    rows = []
    for name, fn in methods.items():
        sec, out = time_fn(fn, repeats=repeats, warmup=max(5, repeats // 10))
        m_fp16 = output_metrics(out, ref_fp16)
        m_int4 = output_metrics(out, ref_int4)

        rows.append({
            "method": name,
            "ms_per_call": sec * 1000,
            "layer": layer_idx,
            "kv_head": kv_head_idx,
            "n_queries": n_queries,
            "seq_len": int(K.shape[0]),
            "head_dim": int(K.shape[1]),
            "BLOCK_T": selected_bt,

            "pair_bps": pair.report["pair_effective_bits_per_scalar"],
            "raw_INT4_inc_scale_bps": pair.report["raw_INT4_inc_scale_bps"],
            "compression_vs_fp16": pair.report["compression_vs_fp16"],
            "saving_vs_INT4": pair.report["saving_ratio_vs_INT4_inc_scale"],

            "K_bits": pair.report["K_bits"],
            "V_bits": pair.report["V_bits"],
            "K_candidate": pair.report["K_candidate"],
            "V_candidate": pair.report["V_candidate"],
            "K_high3_mode": pair.report["K_high3_mode"],
            "V_high3_mode": pair.report["V_high3_mode"],

            "K_handle_kind": K_handle.kind,
            "V_handle_kind": V_handle.kind,
            "K_handle_mode": K_handle.mode,
            "V_handle_mode": V_handle.mode,
            "handle_prepare_sec": prepare_sec,

            "max_diff_vs_original_fp16": m_fp16["max_abs"],
            "mean_diff_vs_original_fp16": m_fp16["mean_abs"],
            "rel_l2_vs_original_fp16": m_fp16["rel_l2"],
            "cos_vs_original_fp16": m_fp16["cos"],

            "max_diff_vs_dense_int4": m_int4["max_abs"],
            "mean_diff_vs_dense_int4": m_int4["mean_abs"],
            "rel_l2_vs_dense_int4": m_int4["rel_l2"],
            "cos_vs_dense_int4": m_int4["cos"],
        })

    df = pd.DataFrame(rows)
    fp16_ms = df.loc[df.method == "original_fp16_attention", "ms_per_call"].iloc[0]
    int4_ms = df.loc[df.method == "dense_int4_predecoded_attention", "ms_per_call"].iloc[0]
    df["speedup_vs_fp16_attention"] = fp16_ms / df["ms_per_call"]
    df["speedup_vs_dense_int4_attention"] = int4_ms / df["ms_per_call"]

    return df, pd.DataFrame(autotune_rows), pair.report

print("Paper benchmark functions ready.")

## 3. Main Sweep: 3B all-layer × all-KV-head

3B에서 전체 KV pair를 모두 측정합니다.  
빠른 디버그가 필요하면 맨 위 설정에서 `MAX_PAIRS_DEBUG = 8`처럼 바꾸세요.

In [ ]:
# ============================================================
# Run main 3B all-layer/all-KV-head sweep
# ============================================================

paper_rows = []
paper_autotune_rows = []
paper_reports = []

pair_indices = [(li, hi) for li in range(num_layers) for hi in range(num_kv_heads)]
if MAX_PAIRS_DEBUG is not None:
    pair_indices = pair_indices[:int(MAX_PAIRS_DEBUG)]

total = len(pair_indices)
for idx, (li, hi) in enumerate(pair_indices, 1):
    try:
        df_one, df_auto, rep = benchmark_pair_for_paper(
            layer_idx=li,
            kv_head_idx=hi,
            n_queries=Q_PER_KV,
            repeats=PAPER_MAIN_REPEATS,
            autotune_repeats=PAPER_AUTOTUNE_REPEATS,
        )
        paper_rows.append(df_one)
        paper_autotune_rows.append(df_auto)

        r = dict(rep)
        r.update({"layer": li, "kv_head": hi})
        paper_reports.append(r)

        v6_row = df_one[df_one["method"].str.contains("hpxnt_v6")].iloc[0]
        fp16_ms = df_one.loc[df_one.method == "original_fp16_attention", "ms_per_call"].iloc[0]
        int4_ms = df_one.loc[df_one.method == "dense_int4_predecoded_attention", "ms_per_call"].iloc[0]
        print(
            f"[{idx:03d}/{total}] layer={li:02d} kv={hi:02d} "
            f"bps={v6_row['pair_bps']:.3f} comp={v6_row['compression_vs_fp16']:.2f}x "
            f"fp16={fp16_ms:.4f}ms int4={int4_ms:.4f}ms hpxnt={v6_row['ms_per_call']:.4f}ms "
            f"sp_fp16={v6_row['speedup_vs_fp16_attention']:.2f}x "
            f"diff_i4={v6_row['max_diff_vs_dense_int4']:.2e} cos_i4={v6_row['cos_vs_dense_int4']:.6f}"
        )
    except Exception as e:
        print(f"FAILED layer={li} kv={hi}: {type(e).__name__}: {e}")

assert len(paper_rows) > 0, "No successful pair benchmarks."

df_paper_full = pd.concat(paper_rows, ignore_index=True)
df_paper_autotune = pd.concat(paper_autotune_rows, ignore_index=True)
df_pair_reports = pd.DataFrame(paper_reports)

display(df_paper_full)
display(df_pair_reports)

df_paper_full.to_csv(os.path.join(PAPER_OUT_DIR, "paper_3b_full_method_rows.csv"), index=False)
df_paper_autotune.to_csv(os.path.join(PAPER_OUT_DIR, "paper_3b_blockt_autotune.csv"), index=False)
df_pair_reports.to_csv(os.path.join(PAPER_OUT_DIR, "paper_3b_pair_reports.csv"), index=False)

print("Saved paper CSVs to:", PAPER_OUT_DIR)

## 4. Summary Tables

논문 표에 바로 넣을 수 있는 요약표입니다.

In [ ]:
# ============================================================
# Paper summary tables
# FIXED:
#   1) HP-XNT methods with different BLOCK_T are grouped as one method.
#   2) Main experiment sequence length is frozen as MAIN_T_CACHE,
#      so later sequence-length sweep does not corrupt the final sentence.
# ============================================================

def paper_method_group_name(method: str) -> str:
    if str(method).startswith("hpxnt_v6") or "hpxnt_v6" in str(method):
        return "HP-XNT-v6 online"
    if method == "original_fp16_attention":
        return "FP16 dense attention"
    if method == "dense_int4_predecoded_attention":
        return "INT4 predecoded dense attention"
    return str(method)

df_paper_full = df_paper_full.copy()
df_paper_full["method_group"] = df_paper_full["method"].map(paper_method_group_name)

# Raw method table keeps BLOCK_T-specific rows for debugging.
summary_by_raw_method = df_paper_full.groupby("method").agg(
    avg_ms=("ms_per_call", "mean"),
    median_ms=("ms_per_call", "median"),
    p95_ms=("ms_per_call", lambda x: float(np.percentile(x, 95))),
    avg_bps=("pair_bps", "mean"),
    median_bps=("pair_bps", "median"),
    avg_compression_vs_fp16=("compression_vs_fp16", "mean"),
    avg_saving_vs_INT4=("saving_vs_INT4", "mean"),
    avg_speedup_vs_fp16_attention=("speedup_vs_fp16_attention", "mean"),
    median_speedup_vs_fp16_attention=("speedup_vs_fp16_attention", "median"),
    avg_speedup_vs_dense_int4_attention=("speedup_vs_dense_int4_attention", "mean"),
    max_diff_vs_dense_int4=("max_diff_vs_dense_int4", "max"),
    avg_cos_vs_dense_int4=("cos_vs_dense_int4", "mean"),
    min_cos_vs_dense_int4=("cos_vs_dense_int4", "min"),
).reset_index()

# Paper-facing method table groups all HP-XNT BLOCK_T variants.
summary_by_method = df_paper_full.groupby("method_group").agg(
    avg_ms=("ms_per_call", "mean"),
    median_ms=("ms_per_call", "median"),
    p95_ms=("ms_per_call", lambda x: float(np.percentile(x, 95))),
    avg_bps=("pair_bps", "mean"),
    median_bps=("pair_bps", "median"),
    avg_compression_vs_fp16=("compression_vs_fp16", "mean"),
    avg_saving_vs_INT4=("saving_vs_INT4", "mean"),
    avg_speedup_vs_fp16_attention=("speedup_vs_fp16_attention", "mean"),
    median_speedup_vs_fp16_attention=("speedup_vs_fp16_attention", "median"),
    avg_speedup_vs_dense_int4_attention=("speedup_vs_dense_int4_attention", "mean"),
    max_diff_vs_dense_int4=("max_diff_vs_dense_int4", "max"),
    avg_cos_vs_dense_int4=("cos_vs_dense_int4", "mean"),
    min_cos_vs_dense_int4=("cos_vs_dense_int4", "min"),
    num_rows=("ms_per_call", "count"),
).reset_index()

# Stable order for paper table.
order = {
    "FP16 dense attention": 0,
    "INT4 predecoded dense attention": 1,
    "HP-XNT-v6 online": 2,
}
summary_by_method["order"] = summary_by_method["method_group"].map(order).fillna(99)
summary_by_method = summary_by_method.sort_values("order").drop(columns=["order"]).reset_index(drop=True)

print("Paper-facing grouped summary:")
display(summary_by_method)

print("Raw method summary, including BLOCK_T-specific HP-XNT variants:")
display(summary_by_raw_method)

df_hpxnt_only = df_paper_full[df_paper_full["method_group"] == "HP-XNT-v6 online"].copy()

# Freeze main experiment metadata before sequence-length sweep mutates global T_cache.
MAIN_T_CACHE = int(df_hpxnt_only["seq_len"].mode().iloc[0]) if len(df_hpxnt_only) else int(T_cache)
MAIN_Q_PER_KV = int(df_hpxnt_only["n_queries"].mode().iloc[0]) if len(df_hpxnt_only) else int(Q_PER_KV)
MAIN_NUM_PAIRS = int(len(df_hpxnt_only))

paper_final_report_dict = {
    "model": MODEL_NAME_FOR_REPORT,
    "seq_len_config": int(SEQ_LEN),
    "actual_seq_len": MAIN_T_CACHE,
    "q_per_kv": MAIN_Q_PER_KV,
    "num_pairs": MAIN_NUM_PAIRS,
    "avg_bps": df_hpxnt_only["pair_bps"].mean(),
    "median_bps": df_hpxnt_only["pair_bps"].median(),
    "avg_compression_vs_fp16": df_hpxnt_only["compression_vs_fp16"].mean(),
    "avg_saving_vs_INT4": df_hpxnt_only["saving_vs_INT4"].mean(),
    "avg_hpxnt_ms": df_hpxnt_only["ms_per_call"].mean(),
    "median_hpxnt_ms": df_hpxnt_only["ms_per_call"].median(),
    "avg_speedup_vs_fp16_attention": df_hpxnt_only["speedup_vs_fp16_attention"].mean(),
    "median_speedup_vs_fp16_attention": df_hpxnt_only["speedup_vs_fp16_attention"].median(),
    "avg_speedup_vs_dense_int4_attention": df_hpxnt_only["speedup_vs_dense_int4_attention"].mean(),
    "median_speedup_vs_dense_int4_attention": df_hpxnt_only["speedup_vs_dense_int4_attention"].median(),
    "max_diff_vs_dense_int4": df_hpxnt_only["max_diff_vs_dense_int4"].max(),
    "avg_cos_vs_dense_int4": df_hpxnt_only["cos_vs_dense_int4"].mean(),
    "min_cos_vs_dense_int4": df_hpxnt_only["cos_vs_dense_int4"].min(),
    "rowlike_K_rate": (df_hpxnt_only["K_handle_kind"] == "rowlike").mean(),
    "rowlike_V_rate": (df_hpxnt_only["V_handle_kind"] == "rowlike").mean(),
}

paper_final_report = pd.DataFrame([paper_final_report_dict]).T
display(paper_final_report)

# Save fixed summaries.
df_paper_full.to_csv(os.path.join(PAPER_OUT_DIR, "paper_3b_full_method_rows.csv"), index=False)
summary_by_raw_method.to_csv(os.path.join(PAPER_OUT_DIR, "paper_3b_summary_by_raw_method.csv"), index=False)
summary_by_method.to_csv(os.path.join(PAPER_OUT_DIR, "paper_3b_summary_by_method.csv"), index=False)
paper_final_report.to_csv(os.path.join(PAPER_OUT_DIR, "paper_3b_final_report.csv"))

print("Saved fixed summary tables.")
print("Main actual seq_len frozen as MAIN_T_CACHE =", MAIN_T_CACHE)

## 5. Paper Figures

그래프는 모두 `PNG + PDF`로 저장됩니다.

출력 위치:

```text
hpxnt_3b_paper_outputs/
```

In [ ]:
# ============================================================
# Clean paper plots
# ============================================================

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

hpxnt = df_paper_full[df_paper_full["method_group"] == "HP-XNT-v6 online"].copy()

# Figure 1: bps heatmap
heat = hpxnt.pivot(index="layer", columns="kv_head", values="pair_bps").sort_index()
plt.figure(figsize=(9, 6))
im = plt.imshow(heat.values, aspect="auto")
plt.colorbar(im, label="Effective bits/scalar")
plt.xlabel("KV head")
plt.ylabel("Layer")
plt.title("HP-XNT effective bits/scalar by layer and KV head")
plt.xticks(range(heat.shape[1]), heat.columns)
plt.yticks(range(heat.shape[0]), heat.index)
savefig_clean("fig1_bps_heatmap_layer_head")
plt.show()

# Figure 2: layer-wise average bps
layer_bps = hpxnt.groupby("layer")["pair_bps"].agg(["mean", "min", "max"]).reset_index()
plt.figure(figsize=(8, 4.5))
plt.plot(layer_bps["layer"], layer_bps["mean"], marker="o", label="Mean bps")
plt.fill_between(layer_bps["layer"], layer_bps["min"], layer_bps["max"], alpha=0.2, label="Min-max range")
plt.axhline(4.0, linestyle="--", linewidth=1, label="INT4 payload")
plt.xlabel("Layer")
plt.ylabel("Effective bits/scalar")
plt.title("Layer-wise HP-XNT compression")
plt.legend()
savefig_clean("fig2_layerwise_bps")
plt.show()

# Figure 3: latency by method
lat = summary_by_method.copy()
plt.figure(figsize=(8, 4.5))
plt.bar(lat["method_group"], lat["avg_ms"])
plt.ylabel("Average latency (ms/call)")
plt.title(f"Attention latency, seq_len={MAIN_T_CACHE}, NQ={MAIN_Q_PER_KV}")
plt.xticks(rotation=20, ha="right")
savefig_clean("fig3_latency_by_method")
plt.show()

# Figure 4: speedup distribution
plt.figure(figsize=(8, 4.5))
plt.hist(hpxnt["speedup_vs_fp16_attention"], bins=20, alpha=0.85)
plt.axvline(hpxnt["speedup_vs_fp16_attention"].mean(), linestyle="--", linewidth=1.5, label="Mean")
plt.xlabel("Speedup vs FP16 attention")
plt.ylabel("Number of layer/head pairs")
plt.title("HP-XNT-v6 speedup distribution")
plt.legend()
savefig_clean("fig4_speedup_histogram")
plt.show()

# Figure 5: compression vs speedup scatter
plt.figure(figsize=(7, 5))
plt.scatter(hpxnt["pair_bps"], hpxnt["speedup_vs_fp16_attention"], alpha=0.8)
plt.xlabel("Effective bits/scalar")
plt.ylabel("Speedup vs FP16 attention")
plt.title("Compression-speed trade-off")
savefig_clean("fig5_compression_vs_speedup")
plt.show()

# Figure 6: HP-XNT vs dense INT4 accuracy
ordered = hpxnt.sort_values(["layer", "kv_head"])
plt.figure(figsize=(8, 4.5))
plt.semilogy(np.arange(len(ordered)), ordered["max_diff_vs_dense_int4"].values, marker=".")
plt.xlabel("Layer/head pair index")
plt.ylabel("Max abs diff vs dense INT4")
plt.title("HP-XNT online attention matches dense INT4")
savefig_clean("fig6_diff_vs_dense_int4")
plt.show()

plt.figure(figsize=(8, 4.5))
plt.plot(np.arange(len(ordered)), ordered["cos_vs_dense_int4"].values, marker=".")
plt.xlabel("Layer/head pair index")
plt.ylabel("Cosine vs dense INT4")
plt.title("Output cosine: HP-XNT vs dense INT4")
savefig_clean("fig6b_cos_vs_dense_int4")
plt.show()

# Figure 7: selected BLOCK_T histogram
plt.figure(figsize=(7, 4.5))
bt_counts = hpxnt["BLOCK_T"].value_counts().sort_index()
plt.bar([str(x) for x in bt_counts.index], bt_counts.values)
plt.xlabel("Selected BLOCK_T")
plt.ylabel("Count")
plt.title("Autotuned BLOCK_T choices")
savefig_clean("fig7_blockt_histogram")
plt.show()

print("All main figures saved to:", PAPER_OUT_DIR)

## 6. Sequence Length Sweep

길이가 증가할 때 attention latency와 speedup이 어떻게 바뀌는지 보여주는 그래프입니다.

기본은 시간 절약을 위해 대표 layer/head만 측정합니다.

최종 논문용으로 더 촘촘히 하려면:

```python
SEQ_SWEEP_ALL_PAIRS = True
```

로 바꾸세요.

In [ ]:
# ============================================================
# Sequence length sweep
# ============================================================

SEQ_SWEEP_LENS = [256, 512, 1024]
SEQ_SWEEP_ALL_PAIRS = False
SEQ_SWEEP_REP_LAYERS = [0, max(0, num_layers // 2), num_layers - 1]
SEQ_SWEEP_REP_KV_HEADS = [0]
SEQ_SWEEP_REPEATS = 60

# Keep main metadata stable for final paper report.
MAIN_T_CACHE_BEFORE_SEQ_SWEEP = globals().get("MAIN_T_CACHE", T_cache)

def refresh_cache_for_seq_len(seq_len):
    global inputs, out, pkv_layers, num_layers, num_kv_heads, T_cache, D_head, head_dim

    prompt = ("In a quiet village near the mountains, a young engineer built a small memory machine. " * 256)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=seq_len).to(model.device)
    with torch.no_grad():
        out = model(**inputs, use_cache=True, output_hidden_states=False, return_dict=True)

    pkv_layers = cache_to_layer_kv(out.past_key_values)
    K0, V0 = normalize_cache_layout(pkv_layers[0][0], pkv_layers[0][1])
    num_layers = len(pkv_layers)
    num_kv_heads = K0.shape[1]
    T_cache = K0.shape[2]
    D_head = K0.shape[3]
    head_dim = D_head

    print("refreshed cache seq_len:", T_cache, "layers:", num_layers, "kv_heads:", num_kv_heads)

def run_sequence_length_sweep():
    rows = []

    for sl in SEQ_SWEEP_LENS:
        refresh_cache_for_seq_len(sl)

        if SEQ_SWEEP_ALL_PAIRS:
            pairs = [(li, hi) for li in range(num_layers) for hi in range(num_kv_heads)]
        else:
            layers = sorted(set([x for x in SEQ_SWEEP_REP_LAYERS if 0 <= x < num_layers]))
            pairs = [(li, hi) for li in layers for hi in SEQ_SWEEP_REP_KV_HEADS if hi < num_kv_heads]

        for li, hi in pairs:
            df_one, _, _ = benchmark_pair_for_paper(
                layer_idx=li,
                kv_head_idx=hi,
                n_queries=Q_PER_KV,
                repeats=SEQ_SWEEP_REPEATS,
                autotune_repeats=max(10, SEQ_SWEEP_REPEATS // 3),
            )
            df_one["sweep_seq_len"] = sl
            rows.append(df_one)

            v6_row = df_one[df_one.method.str.contains("hpxnt_v6")].iloc[0]
            print(
                f"seq={sl} layer={li} kv={hi} "
                f"bps={v6_row['pair_bps']:.3f} speedup={v6_row['speedup_vs_fp16_attention']:.2f}x"
            )

    return pd.concat(rows, ignore_index=True)

df_seq_sweep = run_sequence_length_sweep()
display(df_seq_sweep)
df_seq_sweep.to_csv(os.path.join(PAPER_OUT_DIR, "paper_3b_sequence_length_sweep.csv"), index=False)

# Plot sequence length latency
seq_summary = df_seq_sweep.groupby(["sweep_seq_len", "method"]).agg(
    avg_ms=("ms_per_call", "mean"),
    avg_bps=("pair_bps", "mean"),
).reset_index()

plt.figure(figsize=(8, 5))
for method, sub in seq_summary.groupby("method"):
    sub = sub.sort_values("sweep_seq_len")
    plt.plot(sub["sweep_seq_len"], sub["avg_ms"], marker="o", label=method)
plt.xlabel("Sequence length")
plt.ylabel("Average latency (ms/call)")
plt.title("Representative attention latency vs sequence length")
plt.legend()
savefig_clean("fig8_seq_len_latency")
plt.show()

# Plot sequence length speedup
df_seq_hpxnt = df_seq_sweep[df_seq_sweep.method.str.contains("hpxnt_v6")].copy()
seq_h_summary = df_seq_hpxnt.groupby("sweep_seq_len").agg(
    avg_speedup=("speedup_vs_fp16_attention", "mean"),
    avg_bps=("pair_bps", "mean"),
).reset_index()

plt.figure(figsize=(8, 4.5))
plt.plot(seq_h_summary["sweep_seq_len"], seq_h_summary["avg_speedup"], marker="o")
plt.xlabel("Sequence length")
plt.ylabel("Average speedup vs FP16 attention")
plt.title("Representative HP-XNT speedup vs sequence length")
savefig_clean("fig9_seq_len_speedup")
plt.show()

## 7. Paper-ready Sentences

아래 셀은 논문 결과 문장 초안을 자동 출력합니다.

In [ ]:
# ============================================================
# Auto-generate paper-ready result sentences
# FIXED:
#   Uses paper_final_report_dict["actual_seq_len"] instead of global T_cache.
#   This prevents sequence-length sweep from changing the main-result sentence.
# ============================================================

r = paper_final_report_dict

main_seq_len = int(r["actual_seq_len"])
main_q_per_kv = int(r.get("q_per_kv", Q_PER_KV))

avg_bps = float(r["avg_bps"])
avg_comp = float(r["avg_compression_vs_fp16"])
avg_save = float(r["avg_saving_vs_INT4"])
avg_speed = float(r["avg_speedup_vs_fp16_attention"])
avg_speed_i4 = float(r["avg_speedup_vs_dense_int4_attention"])
max_diff_i4 = float(r["max_diff_vs_dense_int4"])
avg_cos_i4 = float(r["avg_cos_vs_dense_int4"])

english_sentence = (
    f"On {MODEL_NAME_FOR_REPORT} with sequence length {main_seq_len}, HP-XNT achieves "
    f"{avg_bps:.3f} effective bits per scalar on average, corresponding to "
    f"{avg_comp:.2f}x compression over FP16 KV cache and {avg_save*100:.2f}% additional saving over INT4+scale. "
    f"The Triton online attention kernel runs {avg_speed:.2f}x faster than FP16 dense attention "
    f"and {avg_speed_i4:.2f}x faster than dense INT4-predecoded attention on average. "
    f"Compared with dense INT4 attention, HP-XNT has maximum output difference {max_diff_i4:.2e} "
    f"and average output cosine {avg_cos_i4:.6f}."
)

korean_sentence = (
    f"{MODEL_NAME_FOR_REPORT} 모델, sequence length {main_seq_len} 조건에서 HP-XNT는 평균 {avg_bps:.3f} bits/scalar를 달성했다. "
    f"이는 FP16 KV cache 대비 {avg_comp:.2f}배 압축이며, INT4+scale 대비 추가로 {avg_save*100:.2f}%를 절약한 것이다. "
    f"Triton online attention 기준으로 FP16 dense attention보다 평균 {avg_speed:.2f}배, "
    f"dense INT4-predecoded attention보다 평균 {avg_speed_i4:.2f}배 빠르게 동작했다. "
    f"dense INT4 attention 대비 최대 출력 차이는 {max_diff_i4:.2e}, 평균 output cosine은 {avg_cos_i4:.6f}이다."
)

print("Paper-ready English:")
print(english_sentence)

print("\n논문용 한국어:")
print(korean_sentence)

with open(os.path.join(PAPER_OUT_DIR, "paper_ready_result_sentence.txt"), "w", encoding="utf-8") as f:
    f.write(english_sentence + "\n\n")
    f.write(korean_sentence + "\n")

print("Saved:", os.path.join(PAPER_OUT_DIR, "paper_ready_result_sentence.txt"))
print("Main sentence seq_len:", main_seq_len)

In [ ]:
# ============================================================
# Zip all paper experiment outputs
# ============================================================

import os
import zipfile
from datetime import datetime

PAPER_OUT_DIR = globals().get("PAPER_OUT_DIR", "hpxnt_3b_paper_outputs")
assert os.path.exists(PAPER_OUT_DIR), f"Output directory not found: {PAPER_OUT_DIR}"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"HP_XNT_Llama3B_paper_results_FIXED_{timestamp}.zip"
zip_path = os.path.join(".", zip_name)

include_exts = {".csv", ".png", ".pdf", ".txt", ".json"}

file_count = 0
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(PAPER_OUT_DIR):
        for file in files:
            ext = os.path.splitext(file)[1].lower()
            if ext in include_exts:
                full_path = os.path.join(root, file)
                arcname = os.path.relpath(full_path, start=".")
                zf.write(full_path, arcname=arcname)
                file_count += 1

print("Created zip:", zip_path)
print("Included files:", file_count)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Colab download skipped:", repr(e))
    print("You can find the zip at:", zip_path)